In [1]:
from itertools import combinations
import os
import itertools
import pickle

import pandas as pd
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from astropy.stats import circcorrcoef
from matplotlib.lines import Line2D
import matplotlib

from behave_analysis.analyze.filtering_data.filtering_functions import (
    identify_angles,
    filter_video_dataframe,
)


Load tracking data for a single session

In [8]:
JAL1_SEQ3_tracking_data_file = r"Z:\Jasmine_Laurence\Experimental_Data\JAL001\001_seq1_3_2023_03_17T08_38_03\processed_data\fully_processed_tracking_data.pickle"
with open(JAL1_SEQ3_tracking_data_file, "rb") as f:
    tracking_data = pickle.load(f)

# JAL3 SEQ3
JAL1_SEQ3_video_df = r"Z:\Jasmine_Laurence\Experimental_Data\JAL001\001_seq1_3_2023_03_17T08_38_03\processed_data\full_video_dataframe.csv"
seq3_video_df = pl.read_csv(JAL1_SEQ3_video_df)

In [6]:
def extract_xys_of_arena_objects(trackingData, barrier_present: bool) -> dict:
    """Get X,Y coordinates of the points of interest in the arena. These are not angles, but XY points. 
    They are named as angles so that the variable names can be used as a lookup table for the angles generated"""
    if not barrier_present:
        shelter_location = (
            np.mean([trackingData["shelter_loc"][0][0], trackingData["shelter_loc"][1][0]]),
            np.mean([trackingData["shelter_loc"][0][1], trackingData["shelter_loc"][1][1]]),
        )
        return {"hsa": shelter_location}

    if barrier_present:
        shelter_location = (
            np.mean([trackingData["shelter_loc"][0][0], trackingData["shelter_loc"][1][0]]),
            np.mean([trackingData["shelter_loc"][0][1], trackingData["shelter_loc"][1][1]]),
        )
        preflipLocation = trackingData["barrier_loc"][0]
        centerLocation = trackingData["barrier_loc"][2]
        postflipLocation = trackingData["barrier_loc"][1]
        return {
            "hsa": shelter_location,
            "h_bar_centre_a": centerLocation,
            "h_postflipbar_a": postflipLocation,
            "h_preflipbar_a": preflipLocation,
        }
        
def create_grid() -> tuple:
    """Create a grid of bins in the arena."""
    xs = np.arange(0, 1024, 1)  # 1024 is the size of the arena
    ys = np.arange(0, 1024, 1)
    _, xEdges, yEdges = np.histogram2d(xs, ys, bins=300)
    return xEdges, yEdges

def generate_position_points(xEdges, yEdges, session_height, plotPointsGenerated=False) -> list:
    """Generate X,Y coordinates of the center of each bin in the arena."""
    xCoords = (xEdges[1:] + xEdges[:-1]) / 2  # Find center of bins
    yCoords = (yEdges[1:] + yEdges[:-1]) / 2  # Find center of bins
    xCoords, yCoords = remove_points_away_from_center_of_circle(xCoords, yCoords, session_height)
    positionPoints = [[x, y] for x in xCoords for y in yCoords]  # pack coords

    # Plot the points generated for visualisation purposes and debugging
    if plotPointsGenerated:
        plt.scatter(*zip(*positionPoints))
        plt.grid(True)
        plt.xlabel("X Coordinates")
        plt.ylabel("Y Coordinates")
        plt.title("Plot of Coordinate Pairs")
        plt.show()

    return positionPoints

def remove_points_away_from_center_of_circle(x, y, session_height) -> tuple:
    """
    Ensures there are no positions outside of the areana by removing them from the x and y coordinates based
    on the fact that the radius of the arena is 460 pixels.

    TODO:
    + Make this function global
    + Make the radius of the arena a variable not hard coded
    """

    dist = np.sqrt(
        ((x - session_height / 2) ** 2) + ((y - session_height / 2) ** 2)
    )  # Use the euclidean distance formula to find the distance from the center of the arena
    filtX = x[dist < 460]  # 460 is size of arena circle radius, see register
    filtY = y[dist < 460]
    return filtX, filtY

def compute_the_angle_between_the_head_and_a_point(headPositionDf, pointOfInterest):
    """
    Input:
    + headPosition, (X, Y) coordinates of the head as well as synthetic head direction
    + pointname: the column name of the point you want to compute the angle to in the tracking data e.g barrier location, etc#
    + idx: Some of the columns have multiple index points e.g barrier location has two points so you need to index which one you want

    Expects:
    - angles to be in radians

    Logic:
    - xLen and yLen are the x and y components of the triangle formed by the head and the point of interest
    - Inverse of Tan is used to obtain the angle in radians between the x and y components, arcTan2 is used to ensure 
    the correct quadrant is returned (Unsure why negative is needed)
    - The pos and negative logic is to covert a 270 degree turn into a -90 degree turn
    - Rotate the coordinate plane by 90 degrees to ensure it is in the same coordinate system as the head direction
    - Ensure angle generated is (from pi to -pi) by wrapping values over 180 degrees back to negative
    """

    # Use the inverse of tan to compute the angle between the head and the point of interest
    xLen = -headPositionDf["xY"].apply(lambda x: x[0]) + pointOfInterest[0]
    yLen = -headPositionDf["xY"].apply(lambda x: x[1]) + pointOfInterest[1]
    angleOFInterest = -np.arctan2(yLen, xLen)

    # Ensure a 270 degree turn is converted to a -90 degree turn
    isAngleOFInterestPositive = angleOFInterest > 0
    isAngleOFInterestNegative = angleOFInterest < 0
    angleOFInterest[isAngleOFInterestNegative] += np.pi
    angleOFInterest[isAngleOFInterestPositive] -= np.pi

    # Ensure angle generated is (from pi to -pi)
    adjustedAngleOfInterest = np.pi + (
        angleOFInterest - headPositionDf["hDir"]
    )  # brackets for order of operations its not atuple
    adjustedAngleOfInterest[adjustedAngleOfInterest > np.pi] = adjustedAngleOfInterest[
        adjustedAngleOfInterest > np.pi
    ] - (2 * np.pi)

    return adjustedAngleOfInterest

def generate_hdir_data_for_each_position_point(positionPoints) -> pd.DataFrame:
    """
    Create synthetic head direction data for each position point in the arena.

    Generate a dataframe of all possible combinations of position points and spins.
    This is the function that generates the synthetic head direction data for each
    "grid" or bin in the arena generated from the bin_the_data_create_edges function.
    """
    spins = np.arange(-np.pi, np.pi, 0.1)

    # create an array of all possible combinations of position points and spins
    data = np.array(list(itertools.product(positionPoints, spins)), dtype=object)
    dataframe = pd.DataFrame(data, columns=["xY", "hDir"])
    return dataframe

def create_optimal_distributions(trackingData, session_height) -> dict:
    """Generates the optimal sampling distributions for objs of interest

    Returns:
    -- dict: of pdfs for each point of interest
    """
    barrier = True
    pointsOfInterest = extract_xys_of_arena_objects(trackingData, barrier)
    xEdges, yEdges = create_grid()
    positionPoints = generate_position_points(xEdges, yEdges, session_height)
    hdir_df = generate_hdir_data_for_each_position_point(positionPoints)

    # Gen angles between synthetic hdir and points of interest
    dict = {}
    for _, point in enumerate(pointsOfInterest):
        dict[point] = compute_the_angle_between_the_head_and_a_point(hdir_df, pointsOfInterest[point])
    return dict, hdir_df

def save_optimal_as_csv(dict, hdir_df) -> None:
    """Converts dictionary to csv"""
    df = pl.DataFrame(dict)
    arr = hdir_df["hDir"].to_numpy().astype(float)

    # Ensure the length of 'arr' matches the number of rows in 'df'
    if len(arr) != df.height:
        print(f"The length of 'arr' does not match the number of rows in 'df': {len(arr)} vs {df.height}.")
        return

    large = df.hstack([pl.Series("hdir",arr)]).to_pandas()
    path = "optimal_distributions.csv"
    large.to_csv(path)

Create the optimal distribution

In [7]:
optimal_dictionary, hdir_df = create_optimal_distributions(tracking_data, session_height=1024)
save_optimal_as_csv(optimal_dictionary, hdir_df)

In [15]:
def plot_condition_titles(conditions, nrows, columns) -> None:
    """Plot titles and remove the axes from the first
    column of subplots that act as sub titles"""
    for c_counter, c in enumerate(conditions):
        ax = plt.subplot(nrows, columns, c_counter * columns + 1)
        ax.text(1, 0.5, c, rotation="horizontal", va="center", ha="center", fontsize=20)
        ax.set_axis_off()

def load_optimals():
    """Load the optimal angles for synth mouse"""
    return pl.read_csv("optimal_distributions.csv")

def create_all_the_permutations_of_angles(columns) -> list:
    """Permutate a list of angle strings"""
    return list(combinations(columns, 2))

def loop_permutations_apply_circcoeff(combinations, video_df) -> dict:
    """Apply coeff to all permutations of angles"""
    rhoDict = {}
    for angle_set in combinations:
        alpha = np.array(video_df[angle_set[0]].to_numpy())
        beta = np.array(video_df[angle_set[1]].to_numpy())
        rho = circcorrcoef(alpha, beta)
        rhoDict[angle_set] = rho
    return rhoDict

In [25]:
angles = ['hdir', 'hsa', 'h_bar_centre_a', 'h_postflipbar_a', 'h_preflipbar_a']
#conditions = ["shelter_only", "barrier_present"]
conditions = ["barrier_present"]

In [ ]:

def plot_the_circular_rho(video_df, conditions) -> None:
    """Assuming the animal perfectly runs around and samples the entire arean,
    how corrleated are each pairs of angles from the real data vs the optimal distribution?"""
    perms = create_all_the_permutations_of_angles(angles)

    # Create optimal rho dict
    optimals = load_optimals()
    optimal_rho_dic = loop_permutations_apply_circcoeff(perms, optimals)
    optimal_rhos = list(optimal_rho_dic.values())
    x_labels = [f"{perms[i][0]} VS {perms[i][1]}" for i in range(len(perms))]

    # plotting logic
    fig, axs = plt.subplots(
        nrows=len(conditions),
        ncols=2,
        figsize=(24, 6),
        sharey=False,
        sharex=True,
        gridspec_kw={"width_ratios": [1, 8]},
    )
    plot_condition_titles(conditions, len(conditions), 2)
    x = np.arange(len(x_labels))  # the label locations
    labels = ["Sampled Distribution", "Optimal Distribution"]
    colors = ["dimgrey", "lightgreen"]
    legend_elements = [Line2D([0], [0], color=color, lw=4, label=label) for color, label in zip(colors, labels)]
    fig.legend(handles=legend_elements, loc="upper right", fontsize=18)

    for con_i, con in enumerate(conditions):
        # compute data
        data = filter_video_dataframe(video_df, con)
        rho_dic = loop_permutations_apply_circcoeff(perms, data)
        real_rhos = list(rho_dic.values())

        # Plot the bar chart
        axs[con_i, 1].bar(x - 0.1, real_rhos, color="darkgrey", label="Real data", width=0.2)
        axs[con_i, 1].bar(x + 0.1, optimal_rhos, color="lightgreen", label="Optimal data", width=0.2)
        axs[con_i, 1].set_xticks(x, x_labels, fontsize = 16)
        axs[con_i, 1].spines["top"].set_visible(False)
        axs[con_i, 1].spines["right"].set_visible(False)
        axs[con_i, 1].spines["left"].set_visible(False)
        axs[con_i, 1].set_ylabel("Circular rho (ρ)", fontsize=16)
        axs[con_i, 1].axhline(linewidth=1, color="black", linestyle="--")
        axs[con_i, 1].set_yticks([-1, -0.5, 0, 0.5, 1],labels = ['-1', '-0.5', '0', '0.5', '1'], fontsize=16)
        axs[con_i, 1].set_xticklabels(x_labels, rotation=10)

    matplotlib.rc('xtick', labelsize=20)
    plt.show()

In [26]:
def plot_the_circular_rho(video_df, conditions) -> None:
    """Publication-ready: Compare circular rho for sampled vs optimal distributions."""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    perms = create_all_the_permutations_of_angles(angles)

    # Create optimal rho dict
    optimals = load_optimals()
    optimal_rho_dic = loop_permutations_apply_circcoeff(perms, optimals)
    optimal_rhos = list(optimal_rho_dic.values())
    x_labels = [f"{perms[i][0]} vs {perms[i][1]}" for i in range(len(perms))]

    # Publication colors (colorblind-friendly)
    real_color = "#0072B2"      # Blue
    optimal_color = "#E69F00"   # Orange

    # Font sizes
    axis_label_size = 22
    tick_label_size = 18
    legend_size = 20
    title_size = 24

    fig, axs = plt.subplots(
        nrows=len(conditions),
        ncols=2,
        figsize=(22, 7),
        sharey=False,
        sharex=True,
        gridspec_kw={"width_ratios": [1, 8]},
    )
    plot_condition_titles(conditions, len(conditions), 2)
    x = np.arange(len(x_labels))  # the label locations

    legend_elements = [
        Line2D([0], [0], color=real_color, lw=8, label="Sampled Distribution"),
        Line2D([0], [0], color=optimal_color, lw=8, label="Optimal Distribution"),
    ]
    fig.legend(
        handles=legend_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.05),
        ncol=2,
        fontsize=legend_size,
        frameon=False,
    )

    for con_i, con in enumerate(conditions):
        # compute data
        data = filter_video_dataframe(video_df, con)
        rho_dic = loop_permutations_apply_circcoeff(perms, data)
        real_rhos = list(rho_dic.values())

        # Plot the bar chart
        axs[con_i, 1].bar(
            x - 0.15, real_rhos, color=real_color, edgecolor='black', label="Sampled", width=0.3
        )
        axs[con_i, 1].bar(
            x + 0.15, optimal_rhos, color=optimal_color, edgecolor='black', label="Optimal", width=0.3
        )
        axs[con_i, 1].set_xticks(x, x_labels, fontsize=tick_label_size)
        axs[con_i, 1].set_ylabel("Circular correlation (ρ)", fontsize=axis_label_size, fontweight='bold')
        axs[con_i, 1].set_ylim(-1.05, 1.05)
        axs[con_i, 1].axhline(0, linewidth=1.5, color="black", linestyle="--")
        axs[con_i, 1].set_yticks([-1, -0.5, 0, 0.5, 1])
        axs[con_i, 1].set_yticklabels(['-1', '-0.5', '0', '0.5', '1'], fontsize=tick_label_size)
        axs[con_i, 1].set_xticklabels(x_labels, rotation=15, ha='right', fontsize=tick_label_size)
        axs[con_i, 1].spines["top"].set_visible(False)
        axs[con_i, 1].spines["right"].set_visible(False)
        axs[con_i, 1].spines["left"].set_linewidth(1.5)
        axs[con_i, 1].spines["bottom"].set_linewidth(1.5)
        axs[con_i, 1].tick_params(width=1.5, length=7)

    plt.tight_layout(pad=0.7, w_pad=0.2, h_pad=0.7)
    plt.subplots_adjust(left=0.08, right=0.98, bottom=0.13, top=0.88)
    plt.show()

In [28]:
def plot_the_circular_rho(video_df, condition="barrier_present"):
    """Publication-ready: Compare circular rho for sampled vs optimal distributions for a single condition."""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D

    perms = create_all_the_permutations_of_angles(angles)

    # Create optimal rho dict
    optimals = load_optimals()
    optimal_rho_dic = loop_permutations_apply_circcoeff(perms, optimals)
    optimal_rhos = list(optimal_rho_dic.values())
    x_labels = [f"{perms[i][0]} vs {perms[i][1]}" for i in range(len(perms))]

    # Publication colors (colorblind-friendly)
    real_color = "#0072B2"      # Blue
    optimal_color = "#E69F00"   # Orange

    # Font sizes
    axis_label_size = 22
    tick_label_size = 18
    legend_size = 20

    fig, ax = plt.subplots(figsize=(14, 6))

    # compute data
    data = filter_video_dataframe(video_df, condition)
    rho_dic = loop_permutations_apply_circcoeff(perms, data)
    real_rhos = list(rho_dic.values())

    x = np.arange(len(x_labels))  # the label locations

    # Plot the bar chart
    ax.bar(
        x - 0.15, real_rhos, color=real_color, edgecolor='black', label="Sampled", width=0.3
    )
    ax.bar(
        x + 0.15, optimal_rhos, color=optimal_color, edgecolor='black', label="Optimal", width=0.3
    )
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=15, ha='right', fontsize=tick_label_size)
    ax.set_ylabel("Circular correlation (ρ)", fontsize=axis_label_size, fontweight='bold')
    ax.set_ylim(-1.05, 1.05)
    ax.axhline(0, linewidth=1.5, color="black", linestyle="--")
    ax.set_yticks([-1, -0.5, 0, 0.5, 1])
    ax.set_yticklabels(['-1', '-0.5', '0', '0.5', '1'], fontsize=tick_label_size)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.5)
    ax.spines["bottom"].set_linewidth(1.5)
    ax.tick_params(width=1.5, length=7)

    legend_elements = [
        Line2D([0], [0], color=real_color, lw=8, label="Sampled Distribution"),
        Line2D([0], [0], color=optimal_color, lw=8, label="Optimal Distribution"),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.12),
        ncol=2,
        fontsize=legend_size,
        frameon=False,
    )

    plt.tight_layout(pad=0.7, w_pad=0.2, h_pad=0.7)
    plt.subplots_adjust(left=0.08, right=0.98, bottom=0.13, top=0.88)
    plt.show()

In [29]:
plot_the_circular_rho(seq3_video_df, conditions)

In [36]:
def plot_circular_rho_heatmap(video_df, condition="barrier_present"):
    """
    Plot a heatmap matrix of circular rho values for all angle pairs for a single condition.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns

    perms = create_all_the_permutations_of_angles(angles)
    optimals = load_optimals()
    optimal_rho_dic = loop_permutations_apply_circcoeff(perms, optimals)
    data = filter_video_dataframe(video_df, condition)
    real_rho_dic = loop_permutations_apply_circcoeff(perms, data)

    # Prepare square matrices for heatmap
    angle_labels = angles
    n = len(angle_labels)
    real_matrix = np.full((n, n), np.nan)
    optimal_matrix = np.full((n, n), np.nan)

    # Fill upper triangle with real, lower with optimal
    for (a, b), rho in real_rho_dic.items():
        i, j = angle_labels.index(a), angle_labels.index(b)
        real_matrix[i, j] = rho
    for (a, b), rho in optimal_rho_dic.items():
        i, j = angle_labels.index(a), angle_labels.index(b)
        optimal_matrix[j, i] = rho

    # Combine for a single heatmap (upper=real, lower=optimal)
    combined = np.where(~np.isnan(real_matrix), real_matrix, optimal_matrix)
    mask = np.zeros_like(combined, dtype=bool)
    mask[np.tril_indices_from(mask, -1)] = True  # mask lower triangle for real
    mask[np.triu_indices_from(mask, 1)] = True   # mask upper triangle for optimal

    fig, ax = plt.subplots(figsize=(8, 7))
    # Plot real (upper triangle)
    sns.heatmap(real_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                xticklabels=angle_labels, yticklabels=angle_labels, mask=np.tril(np.ones_like(real_matrix, dtype=bool), -1),
                cbar=False, ax=ax, annot_kws={"fontsize":14})
    # Plot optimal (lower triangle)
    sns.heatmap(optimal_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                xticklabels=angle_labels, yticklabels=angle_labels, mask=np.triu(np.ones_like(optimal_matrix, dtype=bool), 1),
                cbar=True, ax=ax, annot_kws={"fontsize":14})

    ax.set_title(f"Circular correlation (ρ) matrix: {condition}", fontsize=20, fontweight='bold', pad=20)
    ax.set_xticklabels(angle_labels, rotation=25, ha='right', fontsize=16, fontweight='bold')
    ax.set_yticklabels(angle_labels, rotation=0, fontsize=16, fontweight='bold')
    ax.tick_params(axis='both', which='both', length=0)
    cbar = ax.collections[-1].colorbar
    cbar.ax.tick_params(labelsize=14)
    cbar.set_label("Circular ρ", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # save the heatmap as an eps
    save_path = r"Z:\Laurence\thesis\figures\two_edge_paradigm"
    
    fig.savefig(
        os.path.join(save_path, f"circular_rho_heatmap.eps"),
        format="eps",
        bbox_inches="tight",
        dpi=300,
    )


In [37]:
plot_circular_rho_heatmap(seq3_video_df, condition="barrier_present")